# 08 · Data-preparation audit
Documents what deduplication removes and why a global time split is not used, using the cached **raw** data. It trains nothing and takes a few minutes.

1. What deduplication removes: exact duplicates and conflicting-label rows, per class, separately.
2. Which labels share identical feature vectors.
3. For InSDN DDoS: are the removed copies separate flows on the wire (different times, hosts) with identical measured features?
4. Under the per-class chronological split on raw data: how many test flows also occur in training, and how many test rows the published rule drops because of conflicting labels.
5. Time coverage per class, and whether one global time cut would leave classes absent from training.

Outputs go to `results/audit/`. **Send me that folder (it is small).**

In [ ]:
import sys, os, json, warnings
sys.path.insert(0, os.path.abspath('..'))
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
pd.set_option('display.width', 220, 'display.max_columns', 60, 'display.max_rows', 200)
from sdnids_bench.config import load_config, mode_params, active_datasets, enable_determinism
cfg = load_config('..')
mp = mode_params(cfg)
print('mode:', cfg['mode'], '| results:', cfg['out_dir'])
from sdnids_bench.experiments import load_clean
from sdnids_bench import audit as AU
OUT = os.path.join(cfg['out_dir'], 'audit'); os.makedirs(OUT, exist_ok=True)
DATASETS = active_datasets(cfg, 'B1'); print(DATASETS)

## 1–2. Exact duplicates versus conflicting labels

In [ ]:
for ds in DATASETS:
    raw, _ = load_clean(cfg, ds, dedup=False)
    t = AU.duplicate_breakdown(raw); t.to_csv(os.path.join(OUT, f'{ds}_duplicate_breakdown.csv'), index=False)
    c = AU.conflict_pairs(raw); c.to_csv(os.path.join(OUT, f'{ds}_conflict_pairs.csv'), index=False)
    print('=====', ds); display(t); display(c)

## 3. InSDN DDoS: what the duplicates look like

In [ ]:
raw, _ = load_clean(cfg, 'InSDN', dedup=False)
top = AU.most_repeated(raw, 'DDoS', top=10); top.to_csv(os.path.join(OUT, 'InSDN_DDoS_most_repeated.csv'), index=False)
display(top)

## 4. Duplicates across the per-class chronological split (raw data)

In [ ]:
for ds in DATASETS:
    raw, _ = load_clean(cfg, ds, dedup=False)
    t = AU.cross_partition_duplicates(raw); t.to_csv(os.path.join(OUT, f'{ds}_cross_partition.csv'), index=False)
    print('=====', ds); display(t)

## 5. Time coverage and global-cut feasibility

In [ ]:
for ds in DATASETS:
    raw, _ = load_clean(cfg, ds, dedup=False)
    tc = AU.time_coverage(raw); tc.to_csv(os.path.join(OUT, f'{ds}_time_coverage.csv'))
    gc = AU.global_cut_feasibility(raw); gc.to_csv(os.path.join(OUT, f'{ds}_global_cut.csv'))
    print('=====', ds); display(tc); display(gc)